# Train Custom "Leha" Wake Word Models for openWakeWord

This notebook trains **two** custom openWakeWord models for Leha:
1. **`leha`** — single-word wake (fastest to say)
2. **`hey_leha`** — two-word wake (fewer false triggers)

Based on openWakeWord's official `automatic_model_training.ipynb`, customized for Leha.
Uses synthetic TTS speech (Piper) + openWakeWord's training pipeline. No real recordings needed.

**How to use:**
1. Runtime → Change runtime type → **T4 GPU** (free tier)
2. Runtime → **Run all** (~30-45 min total)
3. Download `leha.onnx` and `hey_leha.onnx` from the final cell
4. Copy them into `jarvis_ai/voices/` on your laptop
5. Run `scripts/install_leha_wake_model.ps1`

## 1. Environment Setup

Installs Piper TTS (synthetic sample generator) + openWakeWord training deps. Takes ~5 min.

In [ ]:
# Install Piper sample generator (synthetic TTS for training clips).
# PIN the v2.0.0 TAG: openWakeWord's train.py does
# `from generate_samples import generate_samples` — v1.0.0 has the file but
# not the function, and current master removed the file entirely in a
# restructure. v2.0.0 is the state the openWakeWord notebook targets.
!git clone --branch v2.0.0 --depth 1 https://github.com/rhasspy/piper-sample-generator || git clone --depth 1 https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
# Phonemizers: piper-phonemize has no wheels for Python 3.12 (cross build
# does); espeak-ng system lib covers the espeak_phonemizer path as well.
!apt-get -y -qq install libespeak-ng1 espeak-ng-data espeak-ng
!pip install piper-phonemize || pip install piper-phonemize-cross
!pip install espeak-phonemizer
!pip install webrtcvad

# Install openWakeWord. Clone to oww_repo — a folder literally named
# "openwakeword" in the working directory shadows the installed package and
# breaks `import openwakeword.model`. A plain install also fails on
# Python 3.12 (tflite-runtime has no wheels); we only need ONNX, so install
# without deps and add the runtime explicitly.
!git clone https://github.com/dscripka/openwakeword oww_repo
!pip install -e ./oww_repo --no-deps
# onnxscript: required by modern torch.onnx.export — without it the final
# model export fails with "No module named 'onnxscript'".
!pip install onnxruntime onnx onnxscript

# Training dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install deep-phonemizer==0.0.19
# datasets 2.21 works with the modern pyarrow/numpy already on Kaggle's image
!pip install "datasets==2.21.0"

# Compatibility patches for old training code on a modern image. Plain inline
# Python — heredocs do not work inside notebook cells.
# 1) torchaudio removed set_audio_backend in 2.2+.
# 2) torch.load defaults to weights_only=True since torch 2.6, which refuses
#    the pickled Piper generator model; force weights_only=False there.
import pathlib
import re
import site
import sysconfig

_roots = [pathlib.Path("piper-sample-generator")]
for _r in set(site.getsitepackages() + [sysconfig.get_paths()["purelib"]]):
    for _pkg in ("torch_audiomentations", "speechbrain"):
        _roots.append(pathlib.Path(_r) / _pkg)
for _root in _roots:
    if not _root.exists():
        continue
    for _path in _root.rglob("*.py"):
        try:
            _text = _path.read_text(encoding="utf-8")
        except Exception:
            continue
        _new = _text
        if "torchaudio.set_audio_backend" in _new:
            _new = _new.replace(
                "torchaudio.set_audio_backend",
                'getattr(torchaudio, "set_audio_backend", lambda *a, **k: None)',
            )
        if _root.name == "piper-sample-generator" and "weights_only" not in _new:
            _new = re.sub(r"torch\.load\(([^()]*)\)", r"torch.load(\1, weights_only=False)", _new)
        if _new != _text:
            _path.write_text(_new, encoding="utf-8")
            print("patched", _path)

# Download openWakeWord embedding models (Colab workaround)
import os
os.makedirs("./oww_repo/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./oww_repo/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./oww_repo/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./oww_repo/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./oww_repo/openwakeword/resources/models/melspectrogram.tflite

print("\n=== Setup complete ===")

## 2. Download Training Data (one-time, ~10 min)

Downloads background noise, room impulse responses, and pre-computed openWakeWord features.
Shared across both model trainings, so only run once.

In [ ]:
import os
import sys
import numpy as np
import scipy
from pathlib import Path
from tqdm import tqdm
import datasets
import yaml

In [ ]:
# Room impulse responses (MIT) — for realistic reverb augmentation.
# Streamed from Hugging Face; the connection occasionally drops mid-stream
# ("client has been closed"), so retry a few times, keep whatever downloaded,
# and never let this optional augmentation data kill the run.
output_dir = "./mit_rirs"
os.makedirs(output_dir, exist_ok=True)
for attempt in range(3):
    try:
        rir_dataset = datasets.load_dataset(
            "davidscripka/MIT_environmental_impulse_responses",
            split="train", streaming=True, trust_remote_code=True,
        )
        for row in tqdm(rir_dataset, desc=f"RIRs (try {attempt+1})"):
            try:
                name = row['audio']['path'].split('/')[-1]
                out = os.path.join(output_dir, name)
                if os.path.exists(out):
                    continue
                scipy.io.wavfile.write(out, 16000, (row['audio']['array']*32767).astype(np.int16))
            except Exception:
                continue
        break
    except Exception as e:
        print(f"RIR download attempt {attempt+1} failed: {type(e).__name__}: {e}")
print(f"Downloaded {len(os.listdir(output_dir))} room impulse responses")

In [ ]:
# Background noise (AudioSet) — one shard is enough for solid training
if not os.path.exists("audioset"):
    os.mkdir("audioset")
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -q -O {out_dir} {link}
!cd audioset && tar -xf bal_train09.tar

output_dir = "./audioset_16k"
os.makedirs(output_dir, exist_ok=True)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="AudioSet"):
    try:
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    except Exception:
        continue
print(f"Downloaded {len(os.listdir(output_dir))} background noise clips")

In [ ]:
# Music background (Free Music Archive) ? optional extra negatives
# Some Colab/Hugging Face image combinations fail on this dataset builder.
# That should not kill wake training: AudioSet + ACAV features are still the
# main negative data, so skip FMA gracefully if it errors.
output_dir = "./fma"
os.makedirs(output_dir, exist_ok=True)
try:
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True, trust_remote_code=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    n_hours = 1
    for i in tqdm(range(n_hours*3600//30), desc="FMA"):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
        scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    print(f"Downloaded {len(os.listdir(output_dir))} music clips")
except Exception as e:
    print(f"Skipping optional FMA music negatives: {type(e).__name__}: {e}")


In [ ]:
# Pre-computed openWakeWord features (negative data, ~2,000 hrs ACAV100M)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
# Validation set for false-positive rate estimation (~11 hrs)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("Downloaded pre-computed feature data")

## 3. Training Helper

Defines a reusable function that trains one model for a given phrase. Called twice — once for "leha", once for "hey leha".

In [ ]:
import glob

# Locate generate_samples.py — newer piper-sample-generator versions moved it
# into a subpackage, and openWakeWord imports it by adding this dir to sys.path.
_gs_hits = glob.glob("piper-sample-generator/**/generate_samples.py", recursive=True)
PIPER_DIR = os.path.dirname(_gs_hits[0]) if _gs_hits else "./piper-sample-generator"
_pt_hits = glob.glob("piper-sample-generator/**/*.pt", recursive=True)
PIPER_MODEL = _pt_hits[0] if _pt_hits else "./piper-sample-generator/models/en_US-libritts_r-medium.pt"
print(f"piper generator dir: {PIPER_DIR}")
print(f"piper model: {PIPER_MODEL}")

# torch>=2.6 defaults torch.load to weights_only=True, which refuses the
# pickled Piper generator model (contains deep-phonemizer classes). This
# official escape hatch restores the old behavior for all subprocesses.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

# GPU sanity probe: Kaggle sometimes assigns a P100, which new torch builds
# no longer ship CUDA kernels for ("no kernel image is available"). If a real
# CUDA op fails, hide the GPU so every training subprocess runs on CPU
# (slower but reliable) instead of crashing.
import torch
try:
    _ = (torch.ones(8, device="cuda") @ torch.ones(8, device="cuda").T)
    print("CUDA OK:", torch.cuda.get_device_name(0))
except Exception as _e:
    print(f"CUDA unusable ({_e}); forcing CPU mode for all training steps")
    os.environ["CUDA_VISIBLE_DEVICES"] = ""

# The owner's REAL "Leha" recordings (attached as a private Kaggle dataset).
# Mixing them into the synthetic positives is what teaches the model the
# owner's actual voice and accent — the pure-synthetic v16 model scored his
# real voice below 0.10 and never fired.
REAL_CLIPS_DIRS = (
    glob.glob("/kaggle/input/*/real_leha_clips")
    + glob.glob("/kaggle/input/*/*/real_leha_clips")
)
print("real owner clip dirs:", REAL_CLIPS_DIRS)


def _mix_in_real_clips(target_phrase):
    """Copy the owner's real recordings into the generated positive-clip dirs.

    openWakeWord's --generate_clips writes synthetic positives into dirs with
    'positive' in the name. Real clips are resampled to 16k mono, peak
    normalized, and written 3x each so ~60 real clips carry weight against
    ~2000 synthetic ones.
    """
    if target_phrase != "leha" or not REAL_CLIPS_DIRS:
        return
    import math as _math
    import numpy as _np
    import soundfile as _sf
    from scipy.signal import resample_poly as _rp

    pos_dirs = [
        d for d in glob.glob("**/*positive*", recursive=True)
        if os.path.isdir(d) and glob.glob(os.path.join(d, "*.wav"))
    ]
    if not pos_dirs:
        print("[real-clips] no positive clip dirs found; skipping mix-in")
        return
    real = []
    for src_dir in REAL_CLIPS_DIRS:
        real += glob.glob(os.path.join(src_dir, "*.wav"))
    print(f"[real-clips] mixing {len(real)} real clips x3 into: {pos_dirs}")
    for d in pos_dirs:
        for i, w in enumerate(real):
            try:
                a, sr = _sf.read(w, dtype="float32")
                if a.ndim > 1:
                    a = a[:, 0]
                if sr != 16000:
                    g = _math.gcd(sr, 16000)
                    a = _rp(a, 16000 // g, sr // g)
                peak = float(_np.max(_np.abs(a))) or 1.0
                a = (a / peak * 0.9).astype(_np.float32)
                for k in range(3):
                    _sf.write(os.path.join(d, f"real_owner_{i:04d}_{k}.wav"), a, 16000)
            except Exception as e:
                print(f"[real-clips] skip {os.path.basename(w)}: {e}")


def train_phrase(target_phrase, model_name, n_samples=2000, n_samples_val=1000, steps=10000):
    """Train one openWakeWord model for a target phrase.

    Writes config YAML, generates synthetic clips, augments, trains,
    and saves the .onnx model to my_custom_model/{model_name}.onnx
    """
    print(f"\n{'='*60}")
    print(f"Training model: {model_name} (phrase: '{target_phrase}')")
    print(f"{'='*60}\n")

    # Load default config and customize
    config = yaml.load(open("oww_repo/examples/custom_model.yml", 'r').read(), yaml.Loader)
    config["target_phrase"] = [target_phrase]
    config["model_name"] = model_name
    config["n_samples"] = n_samples
    config["n_samples_val"] = n_samples_val
    config["steps"] = steps
    # Target metrics — reasonable defaults from openWakeWord docs
    config["target_accuracy"] = 0.6
    config["target_recall"] = 0.25
    # Piper synthetic-speech generator location (detected above)
    config["piper_sample_generator_path"] = PIPER_DIR
    config["piper_model_path"] = PIPER_MODEL
    # Data paths (downloaded above) — only include non-empty dirs so a flaky
    # optional download can never break training.
    config["background_paths"] = [p for p in ['./audioset_16k', './fma'] if os.path.isdir(p) and len(os.listdir(p)) > 0]
    config["rir_paths"] = [p for p in ['./mit_rirs'] if os.path.isdir(p) and len(os.listdir(p)) > 0]
    config["false_positive_validation_data_path"] = "validation_set_features.npy"
    config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

    cfg_path = f"{model_name}_config.yaml"
    with open(cfg_path, 'w') as f:
        yaml.dump(config, f)

    # Step 1: Generate synthetic clips via Piper TTS (~5 min)
    print(f"\n[1/3] Generating synthetic '{target_phrase}' clips...")
    !{sys.executable} oww_repo/openwakeword/train.py --training_config {cfg_path} --generate_clips

    # Mix the owner's real recordings into the positives before augmentation
    _mix_in_real_clips(target_phrase)

    # Step 2: Augment clips (noise, reverb, gain variation)
    print(f"\n[2/3] Augmenting clips...")
    !{sys.executable} oww_repo/openwakeword/train.py --training_config {cfg_path} --augment_clips

    # Step 3: Train model (~5-10 min on T4 GPU)
    print(f"\n[3/3] Training model...")
    !{sys.executable} oww_repo/openwakeword/train.py --training_config {cfg_path} --train_model

    onnx_path = f"my_custom_model/{model_name}.onnx"
    if os.path.exists(onnx_path):
        size_kb = os.path.getsize(onnx_path) / 1024
        print(f"\nSUCCESS: {onnx_path} ({size_kb:.0f} KB)")
        return onnx_path
    else:
        print(f"\nFAILED: {onnx_path} not found. Check training output above.")
        return None

print("Helper function defined.")

## 4. Train "leha" Model

Single-word wake. Faster to say, slightly higher false-trigger risk.

In [ ]:
leha_path = train_phrase("leha", "leha", n_samples=2000, n_samples_val=1000, steps=10000)

## 5. Train "hey leha" Model

Two-word wake. More distinctive, fewer false triggers.

In [ ]:
hey_leha_path = train_phrase("hey leha", "hey_leha", n_samples=2000, n_samples_val=1000, steps=10000)

## 6. Verify Models Load

Confirm both trained models can be loaded by openWakeWord and produce predictions.

In [ ]:
import sys
import os
import numpy as np

# The editable openWakeWord install registers at interpreter startup, which
# this already-running kernel missed — put the repo on sys.path directly.
sys.path.insert(0, os.path.abspath("oww_repo"))
from openwakeword.model import Model

models_to_test = []
if leha_path and os.path.exists(leha_path):
    models_to_test.append(leha_path)
if hey_leha_path and os.path.exists(hey_leha_path):
    models_to_test.append(hey_leha_path)

if models_to_test:
    print(f"Loading {len(models_to_test)} models together...")
    model = Model(wakeword_models=models_to_test, inference_framework="onnx")

    # Smoke test: feed 1 second of silence, confirm predict() returns scores
    silence = np.zeros(1280, dtype=np.int16)
    for i in range(13):  # 13 chunks of 80ms = ~1 second
        scores = model.predict(silence)
    print("\nPrediction keys:", list(scores.keys()))
    print("Silence scores:", {k: round(v, 4) for k, v in scores.items()})
    print("\nBoth models load and predict successfully.")
    print("\nNext: download the .onnx files below and copy to jarvis_ai/voices/")
else:
    print("No models found. Check training output for errors.")

## 7. Download Models

Zip both `.onnx` files for easy download, then use the file browser on the left (📁 icon) to download them to your laptop.

In [ ]:
import os
import shutil
import zipfile

# Output dir: /kaggle/working on Kaggle, /content on Colab, cwd elsewhere
if os.path.isdir("/kaggle/working"):
    OUT = "/kaggle/working"
elif os.path.isdir("/content"):
    OUT = "/content"
else:
    OUT = os.getcwd()

# The modern torch exporter writes weights to an external <name>.onnx.data
# file next to the model. Re-save each model as ONE self-contained .onnx so
# a single downloaded file works anywhere.
import onnx

final = []
for src, dst_name in [(leha_path, "leha.onnx"), (hey_leha_path, "hey_leha.onnx")]:
    if src and os.path.exists(src):
        dst = os.path.join(OUT, dst_name)
        try:
            m = onnx.load(src)  # resolves external .onnx.data automatically
            onnx.save(m, dst)   # saves fully self-contained
        except Exception as e:
            print(f"self-contained save failed for {dst_name} ({e}); copying raw files")
            shutil.copy(src, dst)
            if os.path.exists(src + ".data"):
                shutil.copy(src + ".data", dst + ".data")
        final.append(dst)
        print(f"OK {dst} ({os.path.getsize(dst)/1024:.0f} KB)")

# Zip the models (plus any .data companions) for one-click download
with zipfile.ZipFile(os.path.join(OUT, "leha_wake_models.zip"), "w") as z:
    for f in final:
        z.write(f, os.path.basename(f))
        if os.path.exists(f + ".data"):
            z.write(f + ".data", os.path.basename(f) + ".data")

# Sanity check: the standalone copies must load from OUT, away from the
# training dir, so a missing external-data file cannot slip through again.
import sys
sys.path.insert(0, os.path.abspath("oww_repo"))
import numpy as np
from openwakeword.model import Model as _M
_m = _M(wakeword_models=final, inference_framework="onnx")
_s = _m.predict(np.zeros(1280, dtype=np.int16))
print("Standalone models verified:", list(_s.keys()))

# Kaggle saves ALL of /kaggle/working as kernel output — delete the big
# training data so the output is just the models, not gigabytes of audio.
if OUT == "/kaggle/working":
    for junk in ("audioset", "audioset_16k", "fma", "mit_rirs", "oww_repo",
                 "openwakeword", "piper-sample-generator", "my_custom_model",
                 "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
                 "validation_set_features.npy"):
        p = os.path.join(OUT, junk)
        try:
            shutil.rmtree(p) if os.path.isdir(p) else os.path.exists(p) and os.remove(p)
        except Exception as e:
            print(f"cleanup skip {junk}: {e}")

print("\n" + "="*60)
print("DONE! Download leha.onnx and hey_leha.onnx")
print("Then copy them to: D:\\jarvis\\jarvis_ai\\voices\\")
print("="*60)

---

## How to download from Colab

**Option A — File browser:** Click the 📁 folder icon on the left sidebar → navigate to `/content/` → right-click `leha.onnx` → Download. Repeat for `hey_leha.onnx`.

**Option B — Zip:** Download `leha_wake_models.zip` (contains both) and unzip on your laptop.

## After downloading

1. Copy `leha.onnx` and `hey_leha.onnx` to `D:\jarvis\jarvis_ai\voices\`
2. On your laptop run: `scripts\install_leha_wake_model.ps1`
3. Restart Leha
4. Say **"Leha"** or **"Hey Leha"** — instant wake, no cloud.